In [ ]:
import gc, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA_V2, OUTPUT
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir, _notebook_stem
from src.lstm import LSTMModel
from src.metrics import metrics, gain
from src.model3_utils import (
    FEATURE_SETS, TARGET, LOOKBACK,
    precompute_split_structure, build_sequences_from_cache,
    scale_sequences, SequenceDataset,
    detect_device, compute_batch_size,
    train_seq_model, predict_seq,
    hw_predict_aligned, save_seq_run,
    print_config, print_feature_set_summary,
)

- Runtime Recommendations

1. PATIENCE: 25 → 10-12  (cuts wasted no-improvement epochs)
2. LR_PATIENCE: 8 → 5    (LR drops faster → faster convergence)
3. LOOKBACK: 20 → 10     (halves sequence tensor size; less GPU memory)
   Change in src/model3_utils.py: LOOKBACK = 10
4. MAX_EPOCHS: 100 → 50  (FC models converged well under 100 epochs)

## Configuration

In [ ]:
DATA_SETS = ['chro_A', 'chro_B', 'chro_C', 'chro_D']
NOTEBOOK_NAME = _notebook_stem()

SEED = 42
MAX_EPOCHS = 100
PATIENCE = 25
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.1
BASE_LR = 1e-3
BASE_BATCH = 512

# Override feature sets locally if needed:
# FEATURE_SETS = {k: v for k, v in FEATURE_SETS.items() if k in ('4F', '8F')}

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

## GPU auto-detect

In [ ]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

## Train all datasets

In [ ]:
RUN_DIR = make_run_dir(NOTEBOOK_NAME)
print(f'Output directory: {RUN_DIR}')

grand_total_time = 0.0

for dataset_name in DATA_SETS:
    print(f'\n{"#" * 70}')
    print(f'#  Dataset: {dataset_name}')
    print(f'{"#" * 70}')

    # Load data
    df_train = pd.read_parquet(CLEAN_DATA_V2 / f'{dataset_name}_train_v2.parquet')
    df_val   = pd.read_parquet(CLEAN_DATA_V2 / f'{dataset_name}_val_v2.parquet')
    df_test  = pd.read_parquet(CLEAN_DATA_V2 / f'{dataset_name}_test_v2.parquet')
    df_full = pd.concat([df_train, df_val, df_test], ignore_index=True)

    print(f'Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}')
    print(f'Full data for sequences: {len(df_full):,} rows')

    # Analytic benchmark
    hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)
    hw_coef = hw['coef']

    # Precompute split structure (shared across all feature sets)
    print('Precomputing split structure...')
    cache = precompute_split_structure(df_full, df_train, df_val, df_test,
                                       target=TARGET, lookback=LOOKBACK)
    del df_full  # free memory; cache holds the sorted copy

    results_by_fs = {}
    df_sorted_ref = cache['df']

    pbar = tqdm(FEATURE_SETS.items(), desc=f'{dataset_name} feature sets', unit='set')
    for fs_name, feature_cols in pbar:
        pbar.set_postfix_str(fs_name)

        # Build sequences from cache (fast — no re-sort/merge)
        seq_data = build_sequences_from_cache(cache, feature_cols)
        X_tr, y_tr = seq_data['X_train'], seq_data['y_train']
        X_va, y_va = seq_data['X_val'], seq_data['y_val']
        X_te, y_te = seq_data['X_test'], seq_data['y_test']
        test_idx = seq_data['test_indices']

        print_feature_set_summary(fs_name, len(X_tr), len(X_va), len(X_te), feature_cols)

        # Scale
        X_tr_sc, X_va_sc, X_te_sc, scaler = scale_sequences(X_tr, X_va, X_te)

        # DataLoaders
        BATCH_SIZE = compute_batch_size(len(X_tr), CFG['MAX_BATCH'])
        INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5
        print_config(CFG, BATCH_SIZE, INIT_LR, len(X_tr), MAX_EPOCHS, PATIENCE, WARMUP_EPOCHS, LOOKBACK)

        train_loader = DataLoader(SequenceDataset(X_tr_sc, y_tr), batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
        val_loader = DataLoader(SequenceDataset(X_va_sc, y_va), batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=2, pin_memory=True)

        # Model
        model = LSTMModel(
            n_features=len(feature_cols), hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS, dropout=DROPOUT, seed=SEED,
        ).to(DEVICE)
        print(f'  LSTM params: {sum(p.numel() for p in model.parameters()):,}')

        # Train
        train_result = train_seq_model(
            model, train_loader, val_loader,
            device=DEVICE, amp_dtype=AMP_DTYPE, use_amp=USE_AMP,
            max_epochs=MAX_EPOCHS, patience=PATIENCE,
            lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR,
            init_lr=INIT_LR, warmup_epochs=WARMUP_EPOCHS,
            use_tqdm=True, desc=fs_name,
        )

        # Predict & evaluate
        y_pred = predict_seq(train_result['model'], X_te_sc, DEVICE, AMP_DTYPE, USE_AMP)
        _, _, hw_sse = hw_predict_aligned(hw_coef, df_sorted_ref, test_idx)

        met = metrics(y_te, y_pred)
        g = gain(met['SSE'], hw_sse) * 100
        print(f'  {fs_name}: SSE={met["SSE"]:.4f}  RMSE={met["RMSE"]:.6f}  '
              f'Gain={g:+.2f}%  epochs={train_result["epochs"]}  '
              f'time={train_result["training_time"]:.1f}s')

        results_by_fs[fs_name] = {
            'model': train_result['model'],
            'y_pred': y_pred, 'y_true': y_te,
            'test_indices': test_idx,
            'history': train_result['history'],
            'epochs': train_result['epochs'],
            'training_time': train_result['training_time'],
            'scaler': scaler,
        }

        # Free memory
        del X_tr, X_va, X_te, X_tr_sc, X_va_sc, X_te_sc, train_loader, val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── Save results for this dataset ─────────────────────────────────
    dataset_dir = RUN_DIR / dataset_name
    summary = save_seq_run(
        dataset_dir,
        results_by_fs=results_by_fs,
        hw_coef=hw_coef,
        df_sorted=df_sorted_ref,
    )
    print(f'\nMetrics Summary ({dataset_name}):')
    display(summary)

    # ── Dataset summary ───────────────────────────────────────────────
    ds_time = sum(r['training_time'] for r in results_by_fs.values())
    grand_total_time += ds_time
    print(f'\nLSTM on {dataset_name} — training time: {ds_time / 60:.1f} min')
    for fs_name, res in results_by_fs.items():
        met = metrics(res['y_true'], res['y_pred'])
        _, _, hw_sse = hw_predict_aligned(hw_coef, df_sorted_ref, res['test_indices'])
        g = gain(met['SSE'], hw_sse) * 100
        print(f'  {fs_name}: SSE={met["SSE"]:.4f}  Gain={g:+.2f}%  epochs={res["epochs"]}')

    # Free memory before next dataset
    del results_by_fs, df_sorted_ref, df_train, df_val, df_test, cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Grand Summary

In [ ]:
print(f'\n{"=" * 70}')
print(f'LSTM on all datasets — grand total training time: {grand_total_time / 60:.1f} min')
print(f'Results saved to: {RUN_DIR}')
for ds in DATA_SETS:
    print(f'  {ds}: {RUN_DIR / ds}')

## Cleanup

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Grand total training time: {grand_total_time / 60:.1f} min')

if IN_COLAB:
    runtime.unassign()